# Evaluating Your Judge

## Overview

In notebooks 01 and 02 we built LLM-as-Judge and LLM-as-Jury evaluators. But how do we know the judge itself is reliable? A judge can sound confident while being wrong. It can reward verbosity over correctness, or flip its verdict on the same input across runs.

This notebook implements a rigorous judge calibration workflow based on the **evaluation-driven philosophy**: every automated evaluator must be validated against human-labeled ground truth before it is trusted in production.

## Key Principles

- **Binary pass/fail only** — no 1-5 scales. Scales introduce implicit variation that hides actual failure modes.
- **One failure mode per judge** — each judge evaluates exactly ONE thing. Combine results afterward.
- **Calibrate against human labels** — split labeled data into few-shot / dev / test sets and iterate.
- **Measure what matters** — True Positive Rate (TPR) and True Negative Rate (TNR) tell you how accurate your judge is as an estimator of real model error.

## What We'll Cover

1. Define a single failure mode with clear pass/fail criteria
2. Build a focused judge prompt with structured output
3. Split human-labeled benchmark data into few-shot / dev / test
4. Calibrate the judge on the dev set and analyze disagreements
5. Iterate on the judge prompt based on failure patterns
6. Validate on the held-out test set
7. Test judge repeatability across multiple runs
8. Perform open-coding failure analysis on traces
9. Produce a judge scorecard

## Prerequisites
- AWS account with Bedrock access (Claude Sonnet 4.6)
- Python 3.10+
- `judge_benchmark.jsonl` in the same directory

## Setup and Dependencies

In [2]:
import json
import random
import boto3
import pandas as pd
from collections import Counter
from time import sleep

# Bedrock clien1691datsme2509

bedrock = boto3.client('bedrock-runtime')

# Configuration
JUDGE_MODEL = "us.anthropic.claude-sonnet-4-6"
BENCHMARK_PATH = "./judge_benchmark.jsonl"
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print("✅ Setup complete")

✅ Setup complete


## 1. Why Evaluate the Judge?

Consider these real failure modes of LLM judges:

| Judge Failure | What Happens | Consequence |
|---|---|---|
| **Verbosity bias** | Longer responses get "pass" even when wrong | Inflated quality scores |
| **Confidence bias** | Confident-sounding wrong answers get "pass" | Missed hallucinations |
| **Vague rubric** | Judge interprets "good" differently each run | Unstable, unreproducible scores |
| **Multi-criteria overload** | One prompt checks accuracy AND completeness AND tone | Cannot isolate which criterion failed |

The fix is simple: treat your judge like any other model. Give it labeled test data, measure its accuracy, and iterate until it meets your bar.

> *"A vague judge prompt produces vague results."* — Evaluations Prescriptive Guidance

### The Anti-Pattern: Rating Scales

Rating on a scale from 1-5 introduces massive implicit variation. Ask someone to rate on 1-5 vs 3-7 — same scale width, different numbers, different ratings. All of this hides the actual failure state. **Binary pass/fail is far more useful.**

## 2. Define a Failure Mode

Each judge should evaluate **one specific failure mode**. Not "is this response good?" but "does this response contain the correct factual data from the provided context?"

For this notebook, our failure mode is:

> **Factual Accuracy**: Does the model's response contain population or land area numbers that match the provided context data?

### Pass/Fail Definitions

| Verdict | Definition |
|---|---|
| **Pass** | The numerical data in the response matches the context. Approximate values (e.g., "about 2.4 million" for 2,390,125) are acceptable. Additional commentary that doesn't contradict the data is fine. |
| **Fail** | The response contains a specific but wrong number, reverses a comparison, claims data is unavailable when it's in the context, or doesn't answer the question. |

## 3. Build the Judge Prompt

A good judge prompt has:
- **One criterion** — factual accuracy only
- **Clear pass/fail definitions** — no ambiguity
- **Structured output** — JSON with reasoning before verdict (forces the judge to think before deciding)
- **Few-shot examples** — from the training split (added in Section 5)

### What bad judge prompts look like

```
❌ "Is this response good?"
❌ "Rate this response from 1-5"
❌ "Check accuracy, completeness, tone, and helpfulness. Give pass/fail."
```

Each of these is too vague or too broad.

In [3]:
# Judge prompt template — ONE failure mode, binary pass/fail, structured output
JUDGE_PROMPT_TEMPLATE = """You are evaluating whether a model's response contains factually accurate data compared to the provided context.

## Evaluation Criterion: Factual Accuracy
- PASS: The numerical data in the response matches the context. Approximate values (e.g., "about 2.4 million" for 2,390,125) are acceptable. Additional commentary that does not contradict the data is fine.
- FAIL: The response contains a specific but wrong number, reverses a comparison, claims data is unavailable when it is in the context, or does not answer the question asked.

{few_shot_section}

## Now evaluate this response:

**Question:** {question}
**Context:** {context}
**Model Response:** {model_response}

Respond with ONLY this JSON (no other text):
{{
    "reasoning": "<your step-by-step analysis>",
    "label": "<pass or fail>"
}}
"""

print("Judge prompt template defined")
print(f"Template length: {len(JUDGE_PROMPT_TEMPLATE)} chars")

Judge prompt template defined
Template length: 806 chars


## 4. Load and Split the Gold Benchmark

The benchmark file contains human-labeled examples with clear pass/fail verdicts and rationales. We split it into three sets:

| Split | Size | Purpose |
|---|---|---|
| **Few-shot (training)** | ~15% | Examples placed directly in the judge prompt |
| **Dev** | ~45% | Iterative calibration — compare judge to human labels |
| **Test** | ~40% | Final blind validation — never look at during development |

⚠️ **This is a separate split from your main evaluation dataset.** This small labeled set is specifically for building and validating the judge itself.

In [4]:
# Load benchmark
with open(BENCHMARK_PATH, 'r') as f:
    benchmark = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(benchmark)} benchmark examples")
print(f"Label distribution: {Counter(ex['human_label'] for ex in benchmark)}")

# Shuffle and split
random.shuffle(benchmark)

n = len(benchmark)
n_fewshot = max(2, int(n * 0.15))  # ~15% for few-shot
n_test = int(n * 0.40)              # ~40% for test
n_dev = n - n_fewshot - n_test       # remainder for dev

fewshot_set = benchmark[:n_fewshot]
dev_set = benchmark[n_fewshot:n_fewshot + n_dev]
test_set = benchmark[n_fewshot + n_dev:]

print(f"\nSplit sizes:")
print(f"  Few-shot (training): {len(fewshot_set)}")
print(f"  Dev (calibration):   {len(dev_set)}")
print(f"  Test (validation):   {len(test_set)}")

# Verify no data leakage
fewshot_qs = {ex['question'] for ex in fewshot_set}
dev_qs = {ex['question'] for ex in dev_set}
test_qs = {ex['question'] for ex in test_set}
assert not (fewshot_qs & dev_qs), "Data leakage: few-shot overlaps dev"
assert not (fewshot_qs & test_qs), "Data leakage: few-shot overlaps test"
assert not (dev_qs & test_qs), "Data leakage: dev overlaps test"
print("\n✅ No data leakage between splits")

Loaded 24 benchmark examples
Label distribution: Counter({'pass': 15, 'fail': 9})

Split sizes:
  Few-shot (training): 3
  Dev (calibration):   12
  Test (validation):   9

✅ No data leakage between splits


## 5. Calibrate the Judge on the Dev Set

First, we build the few-shot section from our training split, then run the judge on every dev example and compare to human labels.

In [5]:
def build_few_shot_section(examples):
    """Build few-shot examples string from training split."""
    if not examples:
        return ""
    
    lines = ["## Examples for reference:"]
    for i, ex in enumerate(examples, 1):
        lines.append(f"""
### Example {i}:
**Question:** {ex['question']}
**Context:** {ex['context']}
**Model Response:** {ex['model_response']}
**Verdict:** {ex['human_label']}
**Reasoning:** {ex['human_rationale']}""")
    return "\n".join(lines)


def run_judge(question, context, model_response, few_shot_examples):
    """Run the judge on a single example. Returns parsed dict or error."""
    few_shot_section = build_few_shot_section(few_shot_examples)
    
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        question=question,
        context=context,
        model_response=model_response,
        few_shot_section=few_shot_section
    )
    
    response = bedrock.converse(
        modelId=JUDGE_MODEL,
        messages=[{'role': 'user', 'content': [{'text': prompt}]}],
        inferenceConfig={'maxTokens': 500, 'temperature': 0}
    )
    
    text = response['output']['message']['content'][0]['text'].strip()
    
    # Strip markdown fences if present
    if text.startswith("```"):
        lines = text.split("\n")
        lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()
    
    try:
        result = json.loads(text)
        result['label'] = result['label'].strip().lower()
        return result
    except json.JSONDecodeError:
        return {"reasoning": text, "label": "error", "raw": text}


def evaluate_split(split_data, few_shot_examples, split_name="dev"):
    """Run judge on a full split and compare to human labels."""
    results = []
    
    for i, ex in enumerate(split_data):
        print(f"  Evaluating {split_name} {i+1}/{len(split_data)}...", end="\r")
        
        judge_result = run_judge(
            ex['question'], ex['context'], ex['model_response'], few_shot_examples
        )
        
        results.append({
            'question': ex['question'],
            'model_response': ex['model_response'],
            'human_label': ex['human_label'],
            'judge_label': judge_result['label'],
            'judge_reasoning': judge_result.get('reasoning', ''),
            'human_rationale': ex['human_rationale'],
            'agree': ex['human_label'] == judge_result['label']
        })
        
        sleep(0.5)  # Rate limiting
    
    print(f"  {'':50}")  # Clear progress line
    return results


print("Helper functions defined ✅")

Helper functions defined ✅


In [6]:
# Run judge on dev set
print("Running judge on dev set...")
dev_results = evaluate_split(dev_set, fewshot_set, "dev")

# Calculate metrics
df_dev = pd.DataFrame(dev_results)

# True Positive Rate (TPR): Of actual passes, how many did the judge correctly identify?
actual_pass = df_dev[df_dev['human_label'] == 'pass']
tp = len(actual_pass[actual_pass['judge_label'] == 'pass'])
tpr = tp / len(actual_pass) if len(actual_pass) > 0 else 0

# True Negative Rate (TNR): Of actual fails, how many did the judge correctly identify?
actual_fail = df_dev[df_dev['human_label'] == 'fail']
tn = len(actual_fail[actual_fail['judge_label'] == 'fail'])
tnr = tn / len(actual_fail) if len(actual_fail) > 0 else 0

overall_accuracy = df_dev['agree'].mean()

print(f"\n{'='*50}")
print(f"Dev Set Results (n={len(dev_results)})")
print(f"{'='*50}")
print(f"Overall accuracy: {overall_accuracy:.1%}")
print(f"TPR (catches real passes):  {tpr:.1%}  ({tp}/{len(actual_pass)})")
print(f"TNR (catches real fails):   {tnr:.1%}  ({tn}/{len(actual_fail)})")
print(f"{'='*50}")

Running judge on dev set...
                                                    

Dev Set Results (n=12)
Overall accuracy: 91.7%
TPR (catches real passes):  100.0%  (6/6)
TNR (catches real fails):   83.3%  (5/6)


## 6. Analyze Disagreements

The most valuable part of calibration is understanding **where the judge disagrees with humans**. These disagreements are the judge's failure modes — and they tell you exactly how to improve the prompt.

In [7]:
# Show disagreements
disagreements = df_dev[~df_dev['agree']]

if len(disagreements) == 0:
    print("✅ Perfect agreement on dev set!")
else:
    print(f"Found {len(disagreements)} disagreement(s):\n")
    
    for _, row in disagreements.iterrows():
        print(f"Question: {row['question']}")
        print(f"  Model response: {row['model_response'][:80]}...")
        print(f"  Human label:    {row['human_label']}")
        print(f"  Judge label:    {row['judge_label']}")
        print(f"  Human reason:   {row['human_rationale'][:100]}...")
        print(f"  Judge reason:   {row['judge_reasoning'][:100]}...")
        print()

Found 1 disagreement(s):

Question: What is the population of Indianapolis?
  Model response: The population of Indianapolis is approximately 900,000....
  Human label:    fail
  Judge label:    pass
  Human reason:   The actual population is 887,642. Rounding to 900,000 is a 1.4% overstatement. For a factual accurac...
  Judge reason:   The actual population of Indianapolis is 887,642. The model states 'approximately 900,000.' The diff...



## 7. Iterate on the Judge Prompt

Based on the disagreement analysis above, we can refine the judge prompt. Common refinements:

- **Clarify borderline cases** — if the judge is too strict on approximations, add guidance
- **Add edge case handling** — if the judge mishandles comparisons or refusals, add examples
- **Sharpen the rubric** — if the judge conflates criteria, narrow the definition

After refining, re-run on the dev set to measure improvement. **Never look at the test set during this process.**

Below we create a refined prompt and re-evaluate:

In [8]:
# Refined judge prompt — adjust based on disagreement patterns observed above
JUDGE_PROMPT_V2 = """You are evaluating whether a model's response contains factually accurate data compared to the provided context.

## Evaluation Criterion: Factual Accuracy

### PASS when:
- The numerical data in the response matches the context exactly
- Reasonable approximations are used (e.g., "about 2.4 million" for 2,390,125)
- The response includes additional true commentary beyond what's in the context
- Minor formatting differences (e.g., 518 vs 518.0)

### FAIL when:
- The response states a specific number that differs from the context (even if close)
- The response reverses a comparison (says A > B when B > A)
- The response claims data is unavailable when the context provides it
- The response does not answer the actual question asked
- The response rounds to a number that misrepresents the data (e.g., 887,642 → 900,000)

### Important edge cases:
- "Approximately 2.4 million" for 2,390,125 → PASS (reasonable rounding)
- "920,000" for 918,915 → FAIL (specific number that's wrong)
- Stating correct data plus unverifiable claims → PASS (judge only the factual data)

{few_shot_section}

## Now evaluate this response:

**Question:** {question}
**Context:** {context}
**Model Response:** {model_response}

Respond with ONLY this JSON (no other text):
{{
    "reasoning": "<your step-by-step analysis>",
    "label": "<pass or fail>"
}}
"""

# Temporarily swap the template
original_template = JUDGE_PROMPT_TEMPLATE
JUDGE_PROMPT_TEMPLATE = JUDGE_PROMPT_V2

print("Running refined judge (v2) on dev set...")
dev_results_v2 = evaluate_split(dev_set, fewshot_set, "dev-v2")

# Restore original for comparison
JUDGE_PROMPT_TEMPLATE = original_template

# Calculate v2 metrics
df_dev_v2 = pd.DataFrame(dev_results_v2)

actual_pass_v2 = df_dev_v2[df_dev_v2['human_label'] == 'pass']
tp_v2 = len(actual_pass_v2[actual_pass_v2['judge_label'] == 'pass'])
tpr_v2 = tp_v2 / len(actual_pass_v2) if len(actual_pass_v2) > 0 else 0

actual_fail_v2 = df_dev_v2[df_dev_v2['human_label'] == 'fail']
tn_v2 = len(actual_fail_v2[actual_fail_v2['judge_label'] == 'fail'])
tnr_v2 = tn_v2 / len(actual_fail_v2) if len(actual_fail_v2) > 0 else 0

accuracy_v2 = df_dev_v2['agree'].mean()

print(f"\n{'='*50}")
print(f"Dev Set Comparison")
print(f"{'='*50}")
print(f"{'Metric':<25} {'v1':>10} {'v2':>10}")
print(f"{'-'*50}")
print(f"{'Overall accuracy':<25} {overall_accuracy:>9.1%} {accuracy_v2:>9.1%}")
print(f"{'TPR (real passes)':<25} {tpr:>9.1%} {tpr_v2:>9.1%}")
print(f"{'TNR (real fails)':<25} {tnr:>9.1%} {tnr_v2:>9.1%}")
print(f"{'='*50}")

# Use the better version going forward
if accuracy_v2 >= overall_accuracy:
    JUDGE_PROMPT_TEMPLATE = JUDGE_PROMPT_V2
    print("\n→ Using v2 (refined) for final validation")
else:
    print("\n→ Keeping v1 (original) for final validation")

Running refined judge (v2) on dev set...
                                                    

Dev Set Comparison
Metric                            v1         v2
--------------------------------------------------
Overall accuracy              91.7%    100.0%
TPR (real passes)            100.0%    100.0%
TNR (real fails)              83.3%    100.0%

→ Using v2 (refined) for final validation


## 8. Final Validation on Test Set

Now we run the calibrated judge on the **held-out test set** that was never seen during development. This gives us the unbiased estimate of judge accuracy.

⚠️ **If test performance is unsatisfactory, collect new labeled data and repeat the process. Do not re-tune on the test set.**

In [9]:
print("Running final validation on test set...")
test_results = evaluate_split(test_set, fewshot_set, "test")

df_test = pd.DataFrame(test_results)

actual_pass_test = df_test[df_test['human_label'] == 'pass']
tp_test = len(actual_pass_test[actual_pass_test['judge_label'] == 'pass'])
tpr_test = tp_test / len(actual_pass_test) if len(actual_pass_test) > 0 else 0

actual_fail_test = df_test[df_test['human_label'] == 'fail']
tn_test = len(actual_fail_test[actual_fail_test['judge_label'] == 'fail'])
tnr_test = tn_test / len(actual_fail_test) if len(actual_fail_test) > 0 else 0

accuracy_test = df_test['agree'].mean()

print(f"\n{'='*50}")
print(f"FINAL TEST SET RESULTS (n={len(test_results)})")
print(f"{'='*50}")
print(f"Overall accuracy: {accuracy_test:.1%}")
print(f"TPR (catches real passes):  {tpr_test:.1%}  ({tp_test}/{len(actual_pass_test)})")
print(f"TNR (catches real fails):   {tnr_test:.1%}  ({tn_test}/{len(actual_fail_test)})")
print(f"{'='*50}")

# Show any test disagreements
test_disagreements = df_test[~df_test['agree']]
if len(test_disagreements) > 0:
    print(f"\nTest set disagreements ({len(test_disagreements)}):")
    for _, row in test_disagreements.iterrows():
        print(f"  • {row['question'][:60]}  human={row['human_label']}  judge={row['judge_label']}")

Running final validation on test set...
                                                    

FINAL TEST SET RESULTS (n=9)
Overall accuracy: 100.0%
TPR (catches real passes):  100.0%  (6/6)
TNR (catches real fails):   100.0%  (3/3)


## 9. Test Judge Repeatability

Even with temperature=0, LLM outputs can vary slightly across runs. A reliable judge should produce the **same verdict on the same input every time**.

We run a subset of examples multiple times and check for label flips.

In [10]:
# Pick a subset for repeatability testing
repeat_subset = dev_set[:5]  # First 5 dev examples
N_RUNS = 3

print(f"Running {N_RUNS} repeated evaluations on {len(repeat_subset)} examples...\n")

repeat_results = {ex['question']: [] for ex in repeat_subset}

for run in range(N_RUNS):
    print(f"  Run {run + 1}/{N_RUNS}...")
    for ex in repeat_subset:
        result = run_judge(ex['question'], ex['context'], ex['model_response'], fewshot_set)
        repeat_results[ex['question']].append(result['label'])
        sleep(0.5)

# Analyze stability
print(f"\n{'='*50}")
print(f"Repeatability Results ({N_RUNS} runs)")
print(f"{'='*50}")

stable_count = 0
for question, labels in repeat_results.items():
    is_stable = len(set(labels)) == 1
    stable_count += is_stable
    status = "✅ Stable" if is_stable else "⚠️ FLIPPED"
    print(f"  {status}: {question[:50]}...")
    if not is_stable:
        print(f"           Labels across runs: {labels}")

repeatability = stable_count / len(repeat_subset)
print(f"\nRepeatability: {repeatability:.0%} ({stable_count}/{len(repeat_subset)} stable)")

Running 3 repeated evaluations on 5 examples...

  Run 1/3...
  Run 2/3...
  Run 3/3...

Repeatability Results (3 runs)
  ✅ Stable: What is the population of Seattle?...
  ✅ Stable: What is the population of Austin?...
  ⚠️ FLIPPED: Which city has more people, Houston or Phoenix?...
           Labels across runs: ['pass', 'fail', 'pass', 'fail', 'pass', 'fail']
  ✅ Stable: What is the population of Indianapolis?...

Repeatability: 60% (3/5 stable)


## 10. Failure Analysis: Open Coding

After running evaluations, the most critical step is **structured failure analysis**. Rather than starting with a predetermined list of failure categories, let the data tell you what's failing.

### The Open Coding Process

1. **Read each failed trace carefully** — write brief, descriptive notes about what went wrong
2. **Cluster similar notes** — group them into failure mode categories
3. **Stop at saturation** — when reviewing additional traces yields no new categories (~100 traces is typically sufficient)

> *"Let the data tell you what is failing. Using a biased starting list means you will miss unexpected failure patterns."*

Below we demonstrate this on the judge's own disagreements:

In [11]:
# Combine all disagreements for open coding
all_results = dev_results + (dev_results_v2 if accuracy_v2 > overall_accuracy else []) + test_results
df_all = pd.DataFrame(all_results)
all_disagreements = df_all[~df_all['agree']].drop_duplicates(subset=['question'])

print(f"Total unique disagreements for analysis: {len(all_disagreements)}\n")

if len(all_disagreements) == 0:
    print("✅ No disagreements to analyze — judge matches human labels perfectly.")
else:
    print("Open Coding Notes:")
    print("=" * 60)
    
    open_codes = []
    for _, row in all_disagreements.iterrows():
        # Categorize the disagreement pattern
        if row['human_label'] == 'pass' and row['judge_label'] == 'fail':
            pattern = "Judge too strict (false negative)"
        elif row['human_label'] == 'fail' and row['judge_label'] == 'pass':
            pattern = "Judge too lenient (false positive)"
        else:
            pattern = "Other"
        
        open_codes.append(pattern)
        print(f"\nQuestion: {row['question'][:60]}...")
        print(f"  Pattern: {pattern}")
        print(f"  Human says: {row['human_label']} — {row['human_rationale'][:80]}...")
        print(f"  Judge says: {row['judge_label']} — {row['judge_reasoning'][:80]}...")
    
    # Cluster
    print(f"\n{'='*60}")
    print("Failure Mode Clusters:")
    for pattern, count in Counter(open_codes).most_common():
        print(f"  {pattern}: {count}")

Total unique disagreements for analysis: 1

Open Coding Notes:

Question: What is the population of Indianapolis?...
  Pattern: Judge too lenient (false positive)
  Human says: fail — The actual population is 887,642. Rounding to 900,000 is a 1.4% overstatement. F...
  Judge says: pass — The actual population of Indianapolis is 887,642. The model states 'approximatel...

Failure Mode Clusters:
  Judge too lenient (false positive): 1


## 11. Judge Scorecard

A summary dashboard of judge quality metrics. Use this to decide whether the judge is ready for production use.

In [12]:
print("=" * 60)
print("  JUDGE SCORECARD")
print("=" * 60)
print()
print(f"  Failure Mode:        Factual Accuracy")
print(f"  Judge Model:         {JUDGE_MODEL}")
print(f"  Benchmark Size:      {len(benchmark)} examples")
print()
print(f"  ┌─────────────────────────────────────────┐")
print(f"  │ Dev Set Accuracy:    {accuracy_v2:.0%}                │")
print(f"  │ Test Set Accuracy:   {accuracy_test:.0%}                │")
print(f"  │ TPR (test):          {tpr_test:.0%}                │")
print(f"  │ TNR (test):          {tnr_test:.0%}                │")
print(f"  │ Repeatability:       {repeatability:.0%}                │")
print(f"  └─────────────────────────────────────────┘")
print()

# Recommendations
print("  Recommendations:")
if accuracy_test >= 0.9 and repeatability >= 0.8:
    print("  ✅ Judge is suitable for automated evaluation")
elif accuracy_test >= 0.75:
    print("  ⚠️ Judge is usable but consider human review for borderline cases")
else:
    print("  ❌ Judge needs further calibration before production use")

if repeatability < 0.8:
    print("  ⚠️ Low repeatability — consider increasing temperature to 0 or adding more few-shot examples")

if tpr_test < tnr_test - 0.2:
    print("  ⚠️ Judge is too strict — missing real passes. Relax the rubric for borderline cases.")
elif tnr_test < tpr_test - 0.2:
    print("  ⚠️ Judge is too lenient — missing real fails. Tighten the rubric.")

print()
print("=" * 60)

  JUDGE SCORECARD

  Failure Mode:        Factual Accuracy
  Judge Model:         us.anthropic.claude-sonnet-4-6
  Benchmark Size:      24 examples

  ┌─────────────────────────────────────────┐
  │ Dev Set Accuracy:    100%                │
  │ Test Set Accuracy:   100%                │
  │ TPR (test):          100%                │
  │ TNR (test):          100%                │
  │ Repeatability:       60%                │
  └─────────────────────────────────────────┘

  Recommendations:
  ⚠️ Judge is usable but consider human review for borderline cases
  ⚠️ Low repeatability — consider increasing temperature to 0 or adding more few-shot examples



## Key Takeaways

1. **Evaluate your judge before trusting it.** A judge is just another model — it needs its own test set.

2. **Binary pass/fail beats rating scales.** Scales introduce implicit variation that hides failure modes. A series of focused pass/fail tests is far more useful.

3. **One failure mode per judge.** Don't ask a single judge to check accuracy, completeness, and tone. Build separate judges and combine results.

4. **Split your labeled data.** Few-shot examples come from the training split. Iterate on the dev split. Validate on the test split. Never contaminate.

5. **Measure TPR and TNR, not just accuracy.** A judge that always says "pass" has 100% TPR but 0% TNR. You need both.

6. **Test repeatability.** If the judge flips its verdict on the same input, you can't trust it.

7. **Let failure analysis be data-driven.** Read traces, write notes, cluster into categories. Don't start with a predetermined list.

### What's Next?

- Add more failure modes (completeness, reasoning quality) as separate judges
- Scale the benchmark to 100+ examples per failure mode
- Integrate judge validation into your CI/CD pipeline
- See the [Evaluations Prescriptive Guidance](https://github.com/aws-samples/sample-gen-ai-evaluations-workshop) for the full UCAMI lifecycle